# Detectors in GWFast format

In [1]:
folder = '..'

my_detectors = {
    'LLAsharp': { 'lat':30.563, 'long':-90.774, 'xax':242.71636956358617, 'shape':'L', 'psd_path': f'{folder}/psds/Asharp-asd_2026.txt'},
    'LLA+': { 'lat':30.563, 'long':-90.774, 'xax':242.71636956358617, 'shape':'L', 'psd_path': f'{folder}/psds/AplusDesign.txt'},
    'LHAsharp': { 'lat':46.455, 'long':-119.408, 'xax':170.99924234706103, 'shape':'L', 'psd_path': f'{folder}/psds/Asharp-asd_2026.txt'},
    'LHA+': { 'lat':46.455, 'long':-119.408, 'xax':170.99924234706103, 'shape':'L', 'psd_path': f'{folder}/psds/AplusDesign.txt'},
    'LIAsharp': {'lat':19.613, 'long':77.031, 'xax':287.384, 'shape':'L', 'psd_path': f'{folder}/psds/Asharp-asd_2026.txt'},
    'LIA+': {'lat':19.613, 'long':77.031, 'xax':287.384, 'shape':'L', 'psd_path': f'{folder}/psds/AplusDesign.txt'},
    'ETSL': { 'lat': 40.+31./60., 'long': 9.+25./60., 'xax':45., 'shape':'L', 'psd_path': f'{folder}/psds/ET_cryo_15km_asd.txt'},
    'ETMRL45d': {'lat': 50.+43./60.+23./3600., 'long': 5.+55./60.+14./3600., 'xax':90., 'shape':'L', 'psd_path': f'{folder}/psds/ET_cryo_15km_asd.txt'},
    "CE40km_1p5MW_Aplus_coat": {"lat": 46., "long": -125., "xax": 305, "shape": "L", "psd_path": f'{folder}/psds/CE40km_1p5MW_Aplus_coat_strain.txt'},
    "CE40km_1p5MW_aLIGO_coat": {"lat": 46., "long": -125., "xax": 305, "shape": "L", "psd_path": f'{folder}/psds/CE40km_1p5MW_aLIGO_coat_strain.txt'},
    "CE40km_1p0MW_Aplus_coat": {"lat": 46., "long": -125., "xax": 305, "shape": "L", "psd_path": f'{folder}/psds/CE40km_1p0MW_Aplus_coat_strain.txt'},
    "CE40km_1p0MW_aLIGO_coat": {"lat": 46., "long": -125., "xax": 305, "shape": "L", "psd_path": f'{folder}/psds/CE40km_1p0MW_aLIGO_coat_strain.txt'},
    "CE20km_1p5MW_Aplus_coat": {"lat": 29., "long": -94., "xax": 245., "shape": "L", "psd_path": f'{folder}/psds/CE20km_1p5MW_Aplus_coat_strain.txt'},
    "CE20km_1p5MW_aLIGO_coat": {"lat": 29., "long": -94., "xax": 245., "shape": "L", "psd_path": f'{folder}/psds/CE20km_1p5MW_aLIGO_coat_strain.txt'},
    "CE20km_1p0MW_Aplus_coat": {"lat": 29., "long": -94., "xax": 245., "shape": "L", "psd_path": f'{folder}/psds/CE20km_1p0MW_Aplus_coat_strain.txt'},
    "CE20km_1p0MW_aLIGO_coat": {"lat": 29., "long": -94., "xax": 245., "shape": "L", "psd_path": f'{folder}/psds/CE20km_1p0MW_aLIGO_coat_strain.txt'},
    "ETD": {"lat": 43.722376, "long": 9.475483, "xax": 0., "shape": "T", "psd_path": f'{folder}/psds/ET_cryo_10km_asd.txt', "arm_length": 10}
}


## Saving them in json files

In [2]:
import json 
import os
import copy

output_dir = './gwfast_jsons'
    
os.makedirs(output_dir, exist_ok=True)

for det_name, params in my_detectors.items():
    file_path = os.path.join(output_dir, f"{det_name}.json")
    det_dict = {}
    det_dict[det_name] = copy.deepcopy(my_detectors[det_name])  
    with open(file_path, 'w') as fp:
        json.dump(det_dict, fp)

## Converting to ifo files

In [3]:
import numpy as np

output_dir = './ifos'
    
os.makedirs(output_dir, exist_ok=True)

for det_name, params in my_detectors.items():
    # Fetch Coordinates directly in degrees
    lat_deg = params['lat']
    long_deg = params['long']
    bisector_deg = params['xax']

    # Determine opening angle based on geometry shape
    if params['shape'].upper() == 'L':
        opening_angle = 90.0
    elif params['shape'].upper() == 'T':
        opening_angle = 60.0
    else:
        raise ValueError(f"Unknown shape '{params['shape']}' for detector {det_name}")
    
    # Calculate actual X and Y arm azimuths from the bisector
    xax_deg = (bisector_deg - (opening_angle / 2.0)) % 360.0
    yax_deg = (bisector_deg + (opening_angle / 2.0)) % 360.0

    # Infer Arm Length (km)
    if 'arm_length' in params:
        length = params['arm_length']
    elif 'CE40' in det_name:
        length = 40.0
    elif 'CE20' in det_name:
        length = 20.0
    elif '15km' in params.get('psd_path', ''):
        length = 15.0
    elif 'ET' in det_name:
        length = 10.0 
    else:
        # Default to Advanced LIGO length
        length = 4.0 

    # Defaults for unprovided values
    elevation = 0.0  # Meters
    xarm_tilt = 0.0  # Degrees
    yarm_tilt = 0.0  # Degrees

    if 'Asharp' in det_name:
        min_freq = 3.0
    elif 'ET' in det_name:
        min_freq = 1.0
    else:
        min_freq = 5.0
    max_freq = 4096.0 

    # 5. Format the Bilby .ifo string
    ifo_content = (
        f"name = '{det_name}'\n"
        f"minimum_frequency = {min_freq}\n"
        f"maximum_frequency = {max_freq}\n"
        f"length = {length}\n"
        f"latitude = {lat_deg:.8f}\n"
        f"longitude = {long_deg:.8f}\n"
        f"elevation = {elevation:.1f}\n"
        f"xarm_azimuth = {xax_deg:.8f}\n"
        f"yarm_azimuth = {yax_deg:.8f}\n"
        f"xarm_tilt = {xarm_tilt:.8f}\n"
        f"yarm_tilt = {yarm_tilt:.8f}\n"
        f"power_spectral_density_file = '{params['psd_path']}'\n"
    )

    # 6. Write to file
    file_path = os.path.join(output_dir, f"{det_name}.ifo")
    with open(file_path, "w") as f:
        f.write(ifo_content)
    
    print(f"Generated {file_path}")



Generated ./ifos/LLAsharp.ifo
Generated ./ifos/LLA+.ifo
Generated ./ifos/LHAsharp.ifo
Generated ./ifos/LHA+.ifo
Generated ./ifos/LIAsharp.ifo
Generated ./ifos/LIA+.ifo
Generated ./ifos/ETSL.ifo
Generated ./ifos/ETMRL45d.ifo
Generated ./ifos/CE40km_1p5MW_Aplus_coat.ifo
Generated ./ifos/CE40km_1p5MW_aLIGO_coat.ifo
Generated ./ifos/CE40km_1p0MW_Aplus_coat.ifo
Generated ./ifos/CE40km_1p0MW_aLIGO_coat.ifo
Generated ./ifos/CE20km_1p5MW_Aplus_coat.ifo
Generated ./ifos/CE20km_1p5MW_aLIGO_coat.ifo
Generated ./ifos/CE20km_1p0MW_Aplus_coat.ifo
Generated ./ifos/CE20km_1p0MW_aLIGO_coat.ifo
Generated ./ifos/ETD.ifo


## Detector networks

In [4]:
fname = f'networks/4020hA+LI.json'

desired_detectors = ['CE40km_1p5MW_Aplus_coat', 'CE20km_1p5MW_Aplus_coat', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p5MW_Aplus_coat', 'CE20km_1p5MW_Aplus_coat', 'LIA+'])


In [5]:
fname = f'networks/4020hALLI.json'

desired_detectors = ['CE40km_1p5MW_aLIGO_coat', 'CE20km_1p5MW_aLIGO_coat', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p5MW_aLIGO_coat', 'CE20km_1p5MW_aLIGO_coat', 'LIA+'])


In [6]:
fname = f'networks/4020lA+LI.json'

desired_detectors = ['CE40km_1p0MW_Aplus_coat', 'CE20km_1p0MW_Aplus_coat', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p0MW_Aplus_coat', 'CE20km_1p0MW_Aplus_coat', 'LIA+'])


In [7]:
fname = f'networks/4020lALLI.json'

desired_detectors = ['CE40km_1p0MW_aLIGO_coat', 'CE20km_1p0MW_aLIGO_coat', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p0MW_aLIGO_coat', 'CE20km_1p0MW_aLIGO_coat', 'LIA+'])


In [8]:
fname = f'networks/40hA+ETLI.json'

desired_detectors = ['CE40km_1p5MW_Aplus_coat', 'ETD', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p5MW_Aplus_coat', 'ETD', 'LIA+'])


In [9]:
fname = f'networks/40hALETLI.json'

desired_detectors = ['CE40km_1p5MW_aLIGO_coat', 'ETD', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p5MW_aLIGO_coat', 'ETD', 'LIA+'])


In [10]:
fname = f'networks/40lA+ETLI.json'

desired_detectors = ['CE40km_1p0MW_Aplus_coat', 'ETD', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p0MW_Aplus_coat', 'ETD', 'LIA+'])


In [11]:
fname = f'networks/40lALETLI.json'

desired_detectors = ['CE40km_1p0MW_aLIGO_coat', 'ETD', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p0MW_aLIGO_coat', 'ETD', 'LIA+'])


In [12]:
fname = f'networks/4020hA+ET.json'

desired_detectors = ['CE40km_1p5MW_Aplus_coat', 'CE20km_1p5MW_Aplus_coat', 'ETD']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p5MW_Aplus_coat', 'CE20km_1p5MW_Aplus_coat', 'ETD'])


In [13]:
fname = f'networks/40hA+ET.json'

desired_detectors = ['CE40km_1p5MW_Aplus_coat', 'ETD']

# Use a dictionary comprehension to create the new dictionary
my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p5MW_Aplus_coat', 'ETD'])


In [14]:
fname = f'networks/40hA+ET2L.json'

desired_detectors = ['CE40km_1p5MW_Aplus_coat', 'ETSL', 'ETMRL45d']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['CE40km_1p5MW_Aplus_coat', 'ETSL', 'ETMRL45d'])


In [15]:
fname = f'networks/LLLHLIA+.json'

desired_detectors = ['LLA+', 'LHA+', 'LIA+']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['LLA+', 'LHA+', 'LIA+'])


In [16]:
fname = f'networks/LLLHLIA#.json'

desired_detectors = ['LLAsharp', 'LHAsharp', 'LIAsharp']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['LLAsharp', 'LHAsharp', 'LIAsharp'])


In [17]:
fname = f'networks/ET2L.json'

desired_detectors = ['ETSL', 'ETMRL45d']

my_subset_network = {key: my_detectors[key] for key in desired_detectors if key in my_detectors}

# Check the result
print(my_subset_network.keys())

with open(fname, 'w') as fp:
    json.dump(my_subset_network, fp)

dict_keys(['ETSL', 'ETMRL45d'])
